# DVC from zero — the hands-on half

The practical companion to **`dvc_slides.html`**. The deck explains *why*; this notebook makes
it happen on your machine. Run the cells top to bottom.

### What is DVC, in one paragraph

**DVC** is git for files that are too big for git. Git stores every version of every file forever,
so committing a 1 GB dataset ten times leaves a 10 GB repository that every clone downloads in
full — and GitHub refuses files over 100 MB anyway.

DVC moves the data out to ordinary storage (S3, a network drive, a folder) and leaves git a small
text **pointer** holding the file's **fingerprint** — a short string computed from its bytes. The
repository stays tiny, `dvc push` sends the bytes, and `dvc pull` fetches them back. You keep
using git exactly as before; DVC's one job is making sure the big files match the commit you are
on.

| Part | Deck slides | What you do |
|---|---|---|
| 0 · Setup | 18–19 | install DVC, build a throwaway repo |
| 1 · Version data | 21–28 | `dvc add`, remotes, push/pull, time travel |
| 2 · Pipelines | 29–33 | `dvc.yaml`, `dvc repro`, metrics |
| 3 · DVCLive | 34–35 | log metrics from inside training code |
| 4 · Advanced | 41–54 | experiments, run cache, `.dvcignore`, `artifacts:` |
| 5 · The real dataset | 5–8 | version a 39 MB folder of photographs |

**Safe to run.** Everything happens in a throwaway `dvc_demo/` folder created in Part 0, and
there is a cleanup cell at the end. Nothing outside this folder is touched.

Lines starting with `!` are shell commands — Jupyter runs them for you.

<!-- ar -->
<div dir="rtl" lang="ar">

**DVC من الصفر: الجزء العملي**

هذا الدفتر هو الجانب العملي لعرض `dvc_slides.html`. العرض يشرح *لماذا*، وهذا الدفتر يطبّق على جهازك. شغّل الخلايا من الأعلى إلى الأسفل.

**ما هو DVC، في فقرة واحدة**

**DVC** هو git للملفات الأكبر من أن يحملها git. git يحفظ كل نسخة من كل ملف للأبد، فحفظ ملف بيانات بحجم ١ غيغابايت عشر مرات يترك مستودعاً بحجم ١٠ غيغابايت يحمّله كل من ينسخه، و GitHub يرفض الملفات فوق ١٠٠ ميغابايت أصلاً.

DVC ينقل البيانات إلى تخزين عادي (S3، أو قرص شبكة، أو مجلد)، ويترك لـ git **مؤشراً** نصياً صغيراً يحمل **بصمة** الملف، وهي نص قصير محسوب من بايتاته. المستودع يبقى صغيراً، و `dvc push` يرسل البايتات، و `dvc pull` يجلبها. تستمر في استخدام git كما كنت؛ وظيفة DVC الوحيدة أن تتأكد أن الملفات الكبيرة تطابق الـ commit الذي أنت عليه.

| الجزء | شرائح العرض | ماذا تفعل |
|---|---|---|
| ٠ · الإعداد | 18–19 | تثبيت DVC، وبناء مستودع مؤقت |
| ١ · إصدارات البيانات | 21–28 | `dvc add`، والمخازن البعيدة، و push/pull، والرجوع بالزمن |
| ٢ · خطوط العمل | 29–33 | `dvc.yaml`، و `dvc repro`، والمقاييس |
| ٣ · DVCLive | 34–35 | تسجيل المقاييس من داخل كود التدريب |
| ٤ · متقدم | 41–54 | التجارب، وذاكرة التشغيل، و `.dvcignore`، و `artifacts:` |
| ٥ · البيانات الحقيقية | 5–8 | إصدار مجلد صور بحجم ٣٩ ميغابايت |

**آمن للتشغيل.** كل شيء يحدث داخل مجلد مؤقت `dvc_demo/` يُنشأ في الجزء ٠، وفي النهاية خلية تنظيف. لا شيء خارج هذا المجلد يُلمس.

الأسطر التي تبدأ بـ `!` أوامر shell، و Jupyter يشغّلها لك.

</div>

### The one idea to hold on to

A **hash** is a short fingerprint of a file's bytes. DVC puts the *fingerprint* in Git and
keeps the *bytes* somewhere else. Everything below is bookkeeping around that sentence.

<svg width="100%" viewBox="0 0 820 210" xmlns="http://www.w3.org/2000/svg" style="max-width:820px">
  <defs><marker id="p1" markerWidth="9" markerHeight="9" refX="8" refY="3" orient="auto">
    <path d="M0,0 L0,6 L9,3 z" fill="#945dd6"/></marker></defs>
  <rect x="8" y="30" width="230" height="150" rx="10" fill="#f6f8fa" stroke="#945dd6"/>
  <text x="26" y="58" font-family="sans-serif" font-size="15" font-weight="700" fill="#24292f">Git repository</text>
  <text x="26" y="86"  font-family="monospace" font-size="12" fill="#57606a">train.py          2 KB</text>
  <text x="26" y="108" font-family="monospace" font-size="12" fill="#57606a">dvc.yaml          1 KB</text>
  <text x="26" y="130" font-family="monospace" font-size="12" fill="#945dd6">raw.csv.dvc      88 B</text>
  <text x="26" y="160" font-family="sans-serif" font-size="12" fill="#1a7f37">tiny — clones instantly</text>

  <path d="M246 105 H358" stroke="#945dd6" stroke-width="2" marker-end="url(#p1)"/>
  <text x="252" y="97" font-family="sans-serif" font-size="12" fill="#945dd6">points to a hash</text>

  <rect x="366" y="30" width="220" height="150" rx="10" fill="#f6f8fa" stroke="#945dd6"/>
  <text x="384" y="58" font-family="sans-serif" font-size="15" font-weight="700" fill="#24292f">.dvc/cache</text>
  <text x="384" y="86"  font-family="monospace" font-size="12" fill="#57606a">files/md5/aa/0ebaab...</text>
  <text x="384" y="108" font-family="sans-serif" font-size="12" fill="#57606a">the real bytes</text>
  <text x="384" y="136" font-family="sans-serif" font-size="12" fill="#57606a">git-ignored</text>

  <path d="M594 105 H706" stroke="#945dd6" stroke-width="2" marker-end="url(#p1)"/>
  <text x="600" y="97" font-family="monospace" font-size="12" fill="#945dd6">dvc push</text>

  <rect x="714" y="30" width="98" height="150" rx="10" fill="#f6f8fa" stroke="#945dd6"/>
  <text x="732" y="58" font-family="sans-serif" font-size="14" font-weight="700" fill="#24292f">Remote</text>
  <text x="732" y="86"  font-family="monospace" font-size="12" fill="#57606a">S3 / GCS</text>
  <text x="732" y="108" font-family="monospace" font-size="12" fill="#57606a">SSH / NAS</text>
  <text x="732" y="130" font-family="monospace" font-size="12" fill="#57606a">a folder</text>
</svg>

**Git carries the map. DVC carries the treasure.**

<!-- ar -->
<div dir="rtl" lang="ar">

**الفكرة الوحيدة التي يجب أن تتمسك بها**

**الـ hash** بصمة قصيرة لبايتات الملف. DVC يضع *البصمة* في Git ويحفظ *البايتات* في مكان آخر. كل ما يأتي بعد ذلك تنظيم حول هذه الجملة.

الرسم أعلاه: مستودع Git صغير فيه الكود وملف `raw.csv.dvc` بحجم ٨٨ بايت فقط، وهذا المؤشر يشير إلى بصمة. البايتات الحقيقية في `.dvc/cache` وهي مستثناة من git. و `dvc push` ينسخها إلى المخزن البعيد: S3، أو SSH، أو مجرد مجلد.

**Git يحمل الخريطة. و DVC يحمل الكنز.**

</div>

In [1]:
!pip install -q dvc scikit-learn pandas numpy pyyaml


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تثبيت المكتبات**

بنثبت `dvc` نفسه، زائد scikit-learn و pandas و numpy و pyyaml يلي رح نحتاجها بخطوط العمل.

</div>

`dvc doctor` prints what DVC can see: your OS, Python, and whether the filesystem supports the
shortcuts it likes to use. It is the output to paste when asking for help.

<!-- ar -->
<div dir="rtl" lang="ar">

`dvc doctor` يطبع ما يراه DVC: نظام التشغيل، و Python، وهل نظام الملفات يدعم الاختصارات التي يفضّلها. هذا هو المخرج الذي تلصقه عندما تطلب مساعدة.

</div>

In [2]:
!dvc --version        # a version number here means you are ready
!dvc doctor           # environment report: OS, python, filesystem, remotes

3.67.1
DVC version: 3.67.1 (pip)
-------------------------
Platform: Python 3.12.6 on Linux-6.8.7-2-liquorix-amd64-x86_64-with-glibc2.39
Subprojects:
	dvc_data = 3.18.3
	dvc_objects = 5.2.0
	dvc_render = 1.0.2
	dvc_task = 0.40.2
	scmrepo = 3.6.2
Supports:
	http (aiohttp = 3.13.2, aiohttp-retry = 2.9.1),
	https (aiohttp = 3.13.2, aiohttp-retry = 2.9.1)
Config:
	Global: /home/shamaseen/.config/dvc
	System: /etc/xdg/xdg-ubuntu/dvc


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · التحقق من DVC**

بنطبع نسخة DVC، ولو طلع رقم يعني جاهزين. بعدين `dvc doctor` بيطبع تقرير عن البيئة: النظام، وبايثون، ونظام الملفات، والمخازن البعيدة المدعومة.

</div>

## Step 0.2 · Create the sandbox

We make a fresh folder that we can delete later without regret:

```
dvc_demo/       <- our fake "project", a real git repo
dvc_remote/            <- pretend cloud storage (just a folder on disk)
```

`dvc_remote/` being a plain folder is the point: **a DVC remote is anywhere that can hold
files.** S3, Google Drive, an SSH box, or a directory. The commands are identical.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٠٫٢ · إنشاء مجلد التجربة**

ننشئ مجلداً جديداً نستطيع حذفه لاحقاً دون ندم: `dvc_demo/` هو "مشروعنا" الوهمي، وهو مستودع git حقيقي، و `dvc_remote/` تخزين سحابي وهمي، مجرد مجلد على القرص.

كون `dvc_remote/` مجلداً عادياً هو المقصود: **المخزن البعيد لـ DVC هو أي مكان يستطيع حفظ ملفات.** S3، أو Google Drive، أو جهاز SSH، أو مجلد. الأوامر نفسها.

</div>

In [3]:
import os, shutil, pathlib

BASE   = pathlib.Path.cwd()                 # the folder this notebook lives in ("1/")
PROJ   = BASE / "dvc_demo"           # our sandbox git+dvc repo
REMOTE = BASE / "dvc_remote"                # our fake remote storage

# Start clean every time this cell runs, so re-running the notebook is safe.
for p in (PROJ, REMOTE):
    if p.exists():
        shutil.rmtree(p)
PROJ.mkdir(parents=True)
(PROJ / "src").mkdir()                      # our pipeline scripts will go here
REMOTE.mkdir(parents=True)

os.chdir(PROJ)                              # everything from here on happens inside the sandbox
print("working inside:", os.getcwd())

working inside: /home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-free-traning/dvc/dvc_demo


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · مجلد التجربة والمخزن الوهمي**

بنحدد ثلاث مسارات: المجلد الأصلي `BASE`، ومجلد المشروع `dvc_demo`، ومجلد `dvc_remote` يلي بيلعب دور التخزين السحابي. لو موجودين من قبل بنمسحهم عشان نبلش نظيف، وبنعملهم من جديد مع مجلد `src` للسكربتات، وبعدين بننتقل جوا المشروع.

</div>

## Step 0.3 · Make a dataset

Real projects have a CSV that arrived from somewhere. We generate one so the notebook
needs no downloads: 2,000 rows of fake customer data with a `churn` label to predict.

Note `np.random.default_rng(42)` — the **seed**. Same seed, same numbers, every time.
That is the same reproducibility instinct DVC enforces at the project level.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٠٫٣ · صنع بيانات**

المشاريع الحقيقية فيها ملف CSV جاء من مكان ما. نحن نولّد واحداً حتى لا يحتاج الدفتر أي تحميل: ٢٠٠٠ صف من بيانات عملاء وهمية مع تصنيف `churn` نريد توقعه.

لاحظ `np.random.default_rng(42)`، وهي **البذرة**. نفس البذرة، نفس الأرقام، كل مرة. وهي نفس فكرة إعادة الإنتاج التي يفرضها DVC على مستوى المشروع.

</div>

In [4]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)             # seeded -> everyone gets identical data
n = 2000

df = pd.DataFrame({
    "tenure_months":  rng.integers(1, 72, n),
    "monthly_charge": rng.normal(65, 20, n).round(2),
    "support_calls":  rng.poisson(1.2, n),
    "is_premium":     rng.integers(0, 2, n),
})

# A learnable rule + noise, so the model has a real signal to find.
score = (-0.04 * df.tenure_months + 0.03 * df.monthly_charge
         + 0.55 * df.support_calls - 0.8 * df.is_premium + rng.normal(0, 0.8, n))
df["churn"] = (score > score.mean()).astype(int)

os.makedirs("data", exist_ok=True)
df.to_csv("data/raw.csv", index=False)      # this is the file we are about to version

print(df.shape, "rows x cols;  churn rate:", df.churn.mean().round(3))
df.head()

(2000, 5) rows x cols;  churn rate: 0.488


,tenure_months,monthly_charge,support_calls,is_premium,churn
0,7,50.16,2,1,1
1,55,83.49,0,0,0
2,47,65.69,1,1,1
3,32,59.34,3,1,1
4,31,62.88,2,1,0


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · توليد ملف البيانات**

بنولّد 2000 عميل وهمي ببذرة ثابتة، وعمود `churn` محسوب من قاعدة بسيطة مع ضجيج عشان يكون في نمط حقيقي. بعدين بنحفظهم بـ `data/raw.csv`، وهاد هو الملف يلي رح نعمله إصدارات، وبنطبع الحجم ونسبة المغادرة وأول صفوف.

</div>

---
# Part 1 — Versioning data   ·   deck slides 21–28

## Step 1.1 · `git init` — because DVC lives inside Git

DVC is not a replacement for Git. It is a **companion**: Git versions your code and the small
pointer files, DVC versions the big files those pointers point at.

`git config user.email/name` is set **locally** (this sandbox only) because a fresh machine
often has no identity configured, and `git commit` refuses to run without one.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الأول: إصدارات البيانات**

**الخطوة ١٫١ · `git init`، لأن DVC يعيش داخل Git**

DVC ليس بديلاً عن Git، بل **رفيق** له: Git يحفظ إصدارات الكود وملفات المؤشر الصغيرة، و DVC يحفظ إصدارات الملفات الكبيرة التي تشير إليها هذه المؤشرات.

نضبط `git config user.email/name` **محلياً** (لهذا المجلد فقط)، لأن الجهاز الجديد غالباً ليس فيه هوية مضبوطة، و `git commit` يرفض العمل بدونها.

</div>

In [5]:
!git init -q                                        # -q = quiet
!git config user.email "trainee@qafza.local"        # local to THIS repo only
!git config user.name  "Qafza Trainee"

# Python bytecode should never be committed.
open(".gitignore", "w").write("__pycache__/\n")

!git add .gitignore && git commit -q -m "chore: initial commit"
!git log --oneline

# Remember the branch name ("main" or "master" depending on your git version) --
# we come back to it after time-travelling later.
BRANCH = !git rev-parse --abbrev-ref HEAD
BRANCH = BRANCH[0]
print("branch:", BRANCH)

1daf1c7 (HEAD -> master) chore: initial commit
branch: master


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · إنشاء مستودع git**

بنعمل مستودع git جديد، وبنحط إيميل واسم لهاد المستودع بس. بنكتب `.gitignore` بيتجاهل ملفات بايثون المؤقتة، وبنعمل أول commit. وبنحفظ اسم الفرع الحالي (main أو master حسب نسخة git) بمتغير `BRANCH` عشان نرجع له بعد ما نتنقل بالزمن.

</div>

## Step 1.2 · `dvc init`

One command. Look at what it creates — this is the whole DVC installation inside a project.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ١٫٢ · `dvc init`**

أمر واحد. انظر إلى ما ينشئه، فهذا هو تثبيت DVC كاملاً داخل مشروع.

</div>

In [8]:
!dvc init -q
!dvc config core.analytics false     # optional: turn off DVC's usage telemetry

print("--- .dvc/ ---")
!ls -a .dvc
print("--- git sees ---")
!git status --short

--- .dvc/ ---
.  ..  config  .gitignore  tmp
--- git sees ---
A  .dvc/.gitignore
AM .dvc/config
A  .dvcignore
?? data/


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تهيئة DVC**

بنشغّل `dvc init` عشان يجهز DVC جوا المشروع، وبنطفي إحصاءات الاستخدام. بعدين بنعرض محتوى مجلد `.dvc` الجديد، وبنشوف `git status` عشان نلاحظ إنه DVC جهّز ملفاته لـ git لحاله.

</div>

### What each piece is

| Path | What it is | In Git? |
|---|---|---|
| `.dvc/config` | your remotes and settings | **yes** — teammates need it |
| `.dvc/cache/` | the actual bytes of every version of every tracked file | **no** — machine-local |
| `.dvc/.gitignore` | written by DVC so `cache/` stays out of Git | yes |
| `.dvcignore` | like `.gitignore`, but tells DVC what not to scan | yes |

Notice `git status` already shows the config files staged for you. Commit them.

<!-- ar -->
<div dir="rtl" lang="ar">

**ما هو كل جزء**

| المسار | ما هو | في Git؟ |
|---|---|---|
| `.dvc/config` | مخازنك البعيدة وإعداداتك | **نعم**، زملاؤك يحتاجونه |
| `.dvc/cache/` | البايتات الحقيقية لكل نسخة من كل ملف متتبَّع | **لا**، خاص بالجهاز |
| `.dvc/.gitignore` | يكتبه DVC حتى تبقى `cache/` خارج Git | نعم |
| `.dvcignore` | مثل `.gitignore`، لكنه يخبر DVC بما لا يفحصه | نعم |

لاحظ أن `git status` يعرض ملفات الإعداد جاهزة للحفظ. احفظها.

</div>

In [9]:
!git commit -q -m "chore: dvc init"
!git log --oneline

b397cb0 (HEAD -> master) chore: dvc init
1daf1c7 chore: initial commit


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · حفظ ملفات DVC بـ git**

بنعمل commit لملفات DVC يلي انجهزت، وبنعرض سجل الـ commits.

</div>

## Step 1.3 · `dvc add` — the command that matters

This is **the** DVC command. Watch what it does to the folder.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ١٫٣ · `dvc add`: الأمر المهم**

هذا **هو** أمر DVC. شاهد ماذا يفعل بالمجلد.

</div>

In [10]:
!dvc add data/raw.csv

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-f
                                                                                
!
  0%|          |Adding data/raw.csv to cache          0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/shamaseen/Desktop/S0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 30.52file/s]

To track the changes with git, run:

	git add data/raw.csv.dvc data/.gitignore

To enable auto staging, run:

	dvc config core.autostage true


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · `dvc add` على ملف البيانات**

بنشغّل `dvc add data/raw.csv`. هاد الأمر بيحسب بصمة الملف، وبينسخه للذاكرة المؤقتة، وبيعمل ملف مؤشر صغير، وبيخلي git يتجاهل الملف الحقيقي.

</div>

Three things just happened. Let's prove each one.

**(1) A tiny pointer file was created.** Open it — it is human-readable YAML:

<!-- ar -->
<div dir="rtl" lang="ar">

ثلاثة أشياء حدثت للتو. لنثبت كل واحد.

**(١) أُنشئ ملف مؤشر صغير.** افتحه، إنه YAML يقرؤه الإنسان:

</div>

In [11]:
!cat data/raw.csv.dvc

outs:
- md5: 7a8d52bd58dfc8938a3a230f88134a54
  size: 29655
  hash: md5
  path: raw.csv


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · عرض ملف المؤشر**

بنعرض محتوى `data/raw.csv.dvc` بـ `cat`، عشان نشوف إنه ملف نصي صغير فيه البصمة والحجم والمسار.

</div>

`md5` is the fingerprint of the CSV's bytes. `size` is how big it is. `path` is where the
file belongs. That is ~120 bytes of text standing in for the entire dataset — and text is
exactly what Git is good at.

**(2) Git was told to ignore the real file**, so it can never bloat the repo:

<!-- ar -->
<div dir="rtl" lang="ar">

`md5` بصمة بايتات ملف CSV. و `size` حجمه. و `path` مكانه. حوالي ١٢٠ بايت من النص تقوم مقام البيانات كلها، والنص هو بالضبط ما يجيده Git.

**(٢) طُلب من Git أن يتجاهل الملف الحقيقي**، فلا يمكن أن يضخّم المستودع أبداً:

</div>

In [12]:
!cat data/.gitignore

/raw.csv


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · ملف التجاهل**

بنعرض `data/.gitignore` يلي كتبه DVC، وفيه اسم ملف البيانات، يعني git ما رح يحفظه أبداً.

</div>

**(3) The bytes were copied into the cache**, filed under the first two characters of the
hash (a classic trick to avoid one directory with a million files in it):

<!-- ar -->
<div dir="rtl" lang="ar">

**(٣) نُسخت البايتات إلى الذاكرة المؤقتة**، مصنّفة تحت أول حرفين من البصمة (حيلة معروفة لتجنب مجلد واحد فيه مليون ملف):

</div>

In [13]:
!find .dvc/cache -type f | head

.dvc/cache/files/md5/7a/8d52bd58dfc8938a3a230f88134a54


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · الملف جوا الذاكرة المؤقتة**

بندور على الملفات جوا `.dvc/cache` وبنعرض أولها، عشان نشوف إنه البيانات انحفظت هناك باسم مأخوذ من البصمة.

</div>

## Step 1.4 · The two-command habit

Whenever data changes:

```
dvc add <file>     # DVC records the new bytes and updates the pointer
git add <file>.dvc # Git records the new pointer
git commit
```

Say it out loud a few times. It is 90% of daily DVC.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ١٫٤ · عادة الأمرين**

كلما تغيّرت البيانات: `dvc add <file>` ليسجّل DVC البايتات الجديدة ويحدّث المؤشر، ثم `git add <file>.dvc` ليسجّل Git المؤشر الجديد، ثم `git commit`.

قلها بصوت عالٍ عدة مرات. هذه ٩٠٪ من استخدام DVC اليومي.

</div>

In [14]:
!git add data/raw.csv.dvc data/.gitignore
!git commit -q -m "data: add raw.csv v1"
!git log --oneline

# Sanity check: how big is the repo Git is actually carrying?
print("\ncsv on disk:", os.path.getsize("data/raw.csv"), "bytes")
print("what git stores instead:", os.path.getsize("data/raw.csv.dvc"), "bytes")

026e9c2 (HEAD -> master) data: add raw.csv v1
b397cb0 chore: dvc init
1daf1c7 chore: initial commit

csv on disk: 29655 bytes
what git stores instead: 88 bytes


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · حفظ المؤشر بـ git ومقارنة الأحجام**

بنضيف ملف المؤشر وملف التجاهل لـ git وبنعمل commit. بعدين بنطبع حجم ملف CSV الحقيقي وحجم ملف المؤشر يلي git بيحفظه بداله، عشان نشوف الفرق.

</div>

## Step 1.5 · A remote, and `dvc push`

A **remote** is where the cache gets shared from. We use a local folder here; in production
you would write `s3://bucket/path`, `gs://…`, `ssh://…`, or `gdrive://…` instead. The
commands after this point do not change at all.

`-d` makes it the *default* remote so plain `dvc push` knows where to go.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ١٫٥ · مخزن بعيد، و `dvc push`**

**المخزن البعيد** هو المكان الذي تُشارك منه الذاكرة المؤقتة. نستخدم هنا مجلداً محلياً؛ وفي الإنتاج تكتب `s3://bucket/path` أو `gs://…` أو `ssh://…` أو `gdrive://…`. الأوامر بعد ذلك لا تتغير أبداً.

`-d` يجعله المخزن *الافتراضي*، فيعرف `dvc push` العادي إلى أين يذهب.

</div>

In [15]:
!dvc remote add -d storage "{REMOTE}"      # {REMOTE} is substituted by Jupyter from Python
!cat .dvc/config

!git add .dvc/config
!git commit -q -m "chore: add default dvc remote"

!dvc push          # upload the cache -> remote.  Like `git push`, but for data.
print("--- what landed in the remote ---")
!find "{REMOTE}" -type f | head

Setting 'storage' as a default remote.
[core]
    analytics = false
    remote = storage
['remote "storage"']
    url = /home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-free-traning/dvc/dvc_remote
Pushing
!
  0% Checking cache in '/home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-f
                                                                                
!
  0% Checking cache in '/home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-f
                                                                                
!
  0%|          |Pushing to local                      0/1 [00:00<?,     ?file/s]
Pushing                                                                         
1 file pushed
--- what landed in the remote ---
/home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-free-traning/dvc/dvc_remote/files/md5/7a/8d52bd58dfc8938a3a230f88134a54


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · إضافة مخزن بعيد والرفع**

بنضيف مجلد `dvc_remote` كمخزن بعيد افتراضي، وبنعرض `.dvc/config` عشان نشوف الإعداد، وبنحفظه بـ git. بعدين `dvc push` بيرفع البيانات من الذاكرة المؤقتة للمخزن البعيد، زي `git push` بس للبيانات، وبنعرض الملفات يلي وصلت هناك.

</div>

## Step 1.6 · Prove it works: delete the data and get it back

This is the moment DVC clicks for most people. We delete the CSV **and** the local cache —
simulating a teammate who has just cloned the repo and has no data at all. Then `dvc pull`.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ١٫٦ · أثبت أنه يعمل: احذف البيانات واسترجعها**

هذه اللحظة التي يفهم فيها معظم الناس DVC. نحذف ملف CSV **و**الذاكرة المؤقتة المحلية، كأننا زميل نسخ المستودع للتو وليس عنده أي بيانات. ثم `dvc pull`.

</div>

In [16]:
import shutil
os.remove("data/raw.csv")          # delete the dataset
shutil.rmtree(".dvc/cache")        # and the local cache: nothing left on this machine

print("raw.csv exists?", os.path.exists("data/raw.csv"))
print("but the pointer survived, because it is in Git:")
!ls data/

raw.csv exists? False
but the pointer survived, because it is in Git:
raw.csv.dvc


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · حذف البيانات والذاكرة المؤقتة**

بنحذف ملف `raw.csv` ومجلد الذاكرة المؤقتة بالكامل، يعني ما ضل ولا نسخة من البيانات على الجهاز. بنطبع إنه الملف مش موجود، بس ملف المؤشر لسا موجود لأنه محفوظ بـ git.

</div>

Now the opposite direction. Delete the data locally, then `dvc pull` brings it back from the
**remote** — the storage that holds the actual bytes. This is what a teammate runs after cloning.

<!-- ar -->
<div dir="rtl" lang="ar">

الآن الاتجاه المعاكس. احذف البيانات محلياً، ثم `dvc pull` يعيدها من **المخزن البعيد**، وهو التخزين الذي يحمل البايتات الحقيقية. هذا ما يشغّله الزميل بعد نسخ المستودع.

</div>

In [17]:
!dvc pull          # remote -> cache -> your folder

print("raw.csv exists?", os.path.exists("data/raw.csv"))
print(pd.read_csv("data/raw.csv").shape, "-- identical rows, byte for byte")

Fetching
!
  0% Checking cache in '/home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-f
                                                                                
!
  0% Checking cache in '/home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-f
                                                                                
!
  0%|          |Fetching from local                   0/1 [00:00<?,     ?file/s]
Fetching                                                                        
Building workspace index                              |1.00 [00:00, 51.9entry/s]
Comparing indexes                                    |3.00 [00:00, 2.87kentry/s]
Applying changes                                      |1.00 [00:00,   218file/s]
A       data/raw.csv
1 file fetched and 1 file added
raw.csv exists? True
(2000, 5) -- identical rows, byte for byte


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · استرجاع البيانات بـ `dvc pull`**

بنشغّل `dvc pull` يلي بيجيب البيانات من المخزن البعيد للذاكرة المؤقتة وبعدين للمجلد. بنطبع إنه الملف رجع، وبنقرأه عشان نشوف إنه نفس عدد الصفوف بالظبط.

</div>

> **Onboarding a teammate is now two commands:** `git clone <repo>` then `dvc pull`.
> They get the code and the *exact* matching data. No Drive links, no "which file did you mean".


## Step 1.7 · Time travel

Now the real payoff. We will:

1. change the dataset (append 500 new rows — "March data arrived"),
2. commit it as **v2**,
3. jump back to **v1**,
4. and jump forward to v2 again.

Remember: `git checkout` moves the *pointers*, `dvc checkout` moves the *data*.
Forgetting the second command is the #1 beginner mistake.

<!-- ar -->
<div dir="rtl" lang="ar">

> **ضم زميل جديد صار أمرين:** `git clone <repo>` ثم `dvc pull`. يحصل على الكود والبيانات المطابقة *بالضبط*. لا روابط Drive، ولا "أي ملف تقصد".

**الخطوة ١٫٧ · الرجوع بالزمن**

الآن الفائدة الحقيقية. سوف:

1. نغيّر البيانات (نضيف ٥٠٠ صف جديد، "وصلت بيانات مارس")،
2. نحفظها كـ **v2**،
3. نرجع إلى **v1**،
4. ثم نتقدم إلى v2 مرة أخرى.

تذكّر: `git checkout` ينقل *المؤشرات*، و `dvc checkout` ينقل *البيانات*. نسيان الأمر الثاني هو خطأ المبتدئين الأول.

</div>

In [18]:
# --- create v2 of the dataset ---
extra = df.sample(500, random_state=1)                    # pretend these are new records
pd.concat([df, extra]).to_csv("data/raw.csv", index=False)

!dvc status          # DVC compares the file's hash to the pointer and notices the change

!dvc add data/raw.csv                       # re-hash, update the pointer
!git add data/raw.csv.dvc
!git commit -q -m "data: add March rows (v2)"
!dvc push                                   # share the new version too

!git log --oneline
print("\nrows now:", len(pd.read_csv("data/raw.csv")))
print("pointer now:")
!grep md5 data/raw.csv.dvc

data/raw.csv.dvc:                                                               
	changed outs:
		modified:           data/raw.csv
 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-f
                                                                                
!
  0%|          |Adding data/raw.csv to cache          0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/shamaseen/Desktop/S0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 30.73file/s]

To track the changes with git, run:

	git add data/raw.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
Pushing
!
  0% Checking cache in '/home/shamaseen/Desktop/Shai/qafza

<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · عمل نسخة تانية من البيانات**

بنضيف 500 صف كأنها بيانات جديدة وبنحفظ الملف. `dvc status` بيلاحظ إنه البصمة تغيرت. بعدين عادة الأمرين: `dvc add` و `git add` للمؤشر، و commit باسم v2، و `dvc push` عشان نشارك النسخة الجديدة. بنطبع عدد الصفوف والبصمة الجديدة.

</div>

Time travel. `git checkout` moves the *pointer files* back to how they were at an older commit;
`dvc checkout` then makes the *data files* match those pointers. Two commands, always in that order.

<!-- ar -->
<div dir="rtl" lang="ar">

الرجوع بالزمن. `git checkout` يعيد *ملفات المؤشر* كما كانت في commit أقدم؛ ثم `dvc checkout` يجعل *ملفات البيانات* تطابق تلك المؤشرات. أمران، دائماً بهذا الترتيب.

</div>

In [17]:
# --- travel back to v1 ---
V1 = !git rev-parse HEAD~1                  # capture the previous commit's id into a Python list
V1 = V1[0]
V2 = !git rev-parse HEAD
V2 = V2[0]
print("v1 =", V1[:8], " v2 =", V2[:8])

!git checkout -q {V1}                       # code + pointers go back
print("\nafter git checkout only ->", len(pd.read_csv("data/raw.csv")), "rows (STILL v2 data!)")

!dvc checkout                               # now the data follows the pointer
print("after dvc checkout      ->", len(pd.read_csv("data/raw.csv")), "rows (this is v1)")

v1 = bfaae784  v2 = 141576aa

after git checkout only -> 2500 rows (STILL v2 data!)
Building workspace index                              |2.00 [00:00,  173entry/s]
Comparing indexes                                    |3.00 [00:00, 2.63kentry/s]
Applying changes                                      |1.00 [00:00,  88.2file/s]
M       data/raw.csv
after dvc checkout      -> 2000 rows (this is v1)


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · الرجوع للنسخة الأولى**

بنحفظ رقمي الـ commit للنسختين v1 و v2. بعدين `git checkout` للنسخة الأولى، وبنعدّ الصفوف: لسا v2 لأنه git رجّع المؤشر بس. بعدين `dvc checkout` وبنعدّ من جديد: هلأ صارت v1 لأنه البيانات لحقت المؤشر.

</div>

See the trap in the output above? After `git checkout` alone the file on disk was still the
**new** data while the pointer said **old**. Only `dvc checkout` reconciles them.

`dvc checkout` is near-instant even for huge files because it *links* from the cache instead
of copying. Ten branches sharing a 50 GB dataset cost you 50 GB once, not 500.

Let's return to the latest version and carry on.

<!-- ar -->
<div dir="rtl" lang="ar">

هل رأيت الفخ في المخرج أعلاه؟ بعد `git checkout` وحده كان الملف على القرص ما زال البيانات **الجديدة** بينما المؤشر يقول **القديمة**. فقط `dvc checkout` يطابقهما.

`dvc checkout` شبه فوري حتى للملفات الضخمة، لأنه *يربط* من الذاكرة المؤقتة بدل النسخ. عشرة فروع تتشارك بيانات بحجم ٥٠ غيغابايت تكلّفك ٥٠ غيغابايت مرة واحدة، وليس ٥٠٠.

لنرجع إلى آخر نسخة ونكمل.

</div>

In [18]:
!git checkout -q {BRANCH}      # back to the branch tip, not a detached commit
!dvc checkout -q
print("back on", BRANCH, "with", len(pd.read_csv("data/raw.csv")), "rows")

back on master with 2500 rows


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · الرجوع لآخر الفرع**

بنرجع لآخر الفرع يلي حفظنا اسمه بـ `BRANCH`، وبعدين `dvc checkout` عشان البيانات تطابق، وبنطبع عدد الصفوف للتأكد.

</div>

---
# Part 2 — Pipelines   ·   deck slides 29–33

Versioning the data is step one. Step two is to stop keeping the recipe in your head.

A **pipeline** is a `dvc.yaml` listing *stages*. Each declares `cmd` (what to run), `deps`
(inputs — if a hash changes the stage is stale), `outs` (outputs DVC tracks for you), `params`
and `metrics`. Then `dvc repro` runs **only the stale stages**.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الثاني: خطوط العمل**

إصدارات البيانات الخطوة الأولى. الخطوة الثانية أن تتوقف عن حفظ الوصفة في رأسك.

**خط العمل** ملف `dvc.yaml` يسرد *مراحل*. كل مرحلة تعلن `cmd` (ماذا تشغّل)، و `deps` (المدخلات، وإذا تغيّرت بصمة أحدها تصبح المرحلة قديمة)، و `outs` (مخرجات يتتبعها DVC لك)، و `params` و `metrics`. ثم `dvc repro` يشغّل **المراحل القديمة فقط**.

</div>

### The shape of what we are about to build

<svg width="100%" viewBox="0 0 830 100" xmlns="http://www.w3.org/2000/svg" style="max-width:830px">
  <defs><marker id="d1" markerWidth="9" markerHeight="9" refX="8" refY="3" orient="auto">
    <path d="M0,0 L0,6 L9,3 z" fill="#945dd6"/></marker></defs>
  <rect x="6"   y="22" width="132" height="52" rx="8" fill="#f6f8fa" stroke="#d0d7de"/>
  <text x="24"  y="45" font-family="monospace" font-size="12" fill="#24292f">data/raw.csv</text>
  <text x="24"  y="64" font-family="sans-serif" font-size="11" fill="#57606a">dvc add</text>
  <path d="M143 48 H196" stroke="#945dd6" stroke-width="2" marker-end="url(#d1)"/>

  <rect x="203" y="22" width="112" height="52" rx="8" fill="#f6f8fa" stroke="#945dd6"/>
  <text x="230" y="45" font-family="monospace" font-size="13" fill="#945dd6">prepare</text>
  <text x="218" y="64" font-family="sans-serif" font-size="11" fill="#57606a">prepare.py</text>
  <path d="M320 48 H373" stroke="#945dd6" stroke-width="2" marker-end="url(#d1)"/>

  <rect x="380" y="22" width="142" height="52" rx="8" fill="#f6f8fa" stroke="#d0d7de"/>
  <text x="396" y="45" font-family="monospace" font-size="12" fill="#24292f">data/prepared/</text>
  <text x="396" y="64" font-family="sans-serif" font-size="11" fill="#57606a">train.csv + test.csv</text>
  <path d="M527 48 H580" stroke="#945dd6" stroke-width="2" marker-end="url(#d1)"/>

  <rect x="587" y="22" width="100" height="52" rx="8" fill="#f6f8fa" stroke="#945dd6"/>
  <text x="612" y="45" font-family="monospace" font-size="13" fill="#945dd6">train</text>
  <text x="600" y="64" font-family="sans-serif" font-size="11" fill="#57606a">+ params.yaml</text>
  <path d="M692 48 H735" stroke="#945dd6" stroke-width="2" marker-end="url(#d1)"/>

  <text x="742" y="43" font-family="monospace" font-size="12" fill="#24292f">model.pkl</text>
  <text x="742" y="62" font-family="monospace" font-size="12" fill="#24292f">metrics.json</text>
</svg>

`dvc repro` walks this graph and runs only the boxes whose inputs changed.

## Step 2.1 · The knobs file, `params.yaml`

Hyperparameters live in a file instead of being buried in the script. DVC watches it, so
changing a number here is enough to make `dvc repro` retrain.

<!-- ar -->
<div dir="rtl" lang="ar">

**شكل ما سنبنيه**

الرسم أعلاه: `data/raw.csv` (متتبَّع بـ `dvc add`) يدخل مرحلة `prepare`، فتنتج `data/prepared/` (ملفا التدريب والاختبار)، وهذا يدخل مرحلة `train` مع `params.yaml`، فتنتج `model.pkl` و `metrics.json`.

`dvc repro` يمشي على هذا الرسم ولا يشغّل إلا الصناديق التي تغيّرت مدخلاتها.

**الخطوة ٢٫١ · ملف الإعدادات `params.yaml`**

المعاملات تعيش في ملف بدل أن تُدفن داخل السكربت. DVC يراقبه، فتغيير رقم هنا يكفي ليعيد `dvc repro` التدريب.

</div>

In [19]:
%%writefile params.yaml
test_size: 0.2
random_state: 42
n_estimators: 60
max_depth: 4

Writing params.yaml


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · ملف المعاملات**

`%%writefile` بتكتب `params.yaml` فيه حجم الاختبار، والبذرة، وعدد الأشجار، والعمق.

</div>

## Step 2.2 · The stage scripts

`%%writefile` dumps the cell's contents to a file instead of executing it. We write two
plain Python scripts — nothing DVC-specific inside them, which is the point: **DVC wraps
ordinary scripts, it does not invade them.**

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٢٫٢ · سكربتات المراحل**

`%%writefile` يكتب محتوى الخلية في ملف بدل تنفيذه. نكتب سكربتين Python عاديين، لا شيء خاص بـ DVC داخلهما، وهذا هو المقصود: **DVC يغلّف السكربتات العادية، ولا يتدخل فيها.**

</div>

In [20]:
%%writefile src/prepare.py
"""Stage 1: read the raw CSV, split into train/test, save both."""
import pandas as pd, yaml, os
from sklearn.model_selection import train_test_split

p = yaml.safe_load(open("params.yaml"))          # read the knobs

df = pd.read_csv("data/raw.csv")
train, test = train_test_split(df, test_size=p["test_size"],
                               random_state=p["random_state"], stratify=df.churn)

os.makedirs("data/prepared", exist_ok=True)
train.to_csv("data/prepared/train.csv", index=False)
test.to_csv("data/prepared/test.csv", index=False)
print(f"prepare: {len(train)} train / {len(test)} test rows")

Writing src/prepare.py


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · سكربت المرحلة الأولى**

`%%writefile` بتكتب `src/prepare.py`: بيقرأ المعاملات والبيانات الخام، وبيقسمها تدريب واختبار بـ `train_test_split`، وبيحفظ الملفين بمجلد `data/prepared`. كود بايثون عادي، ما فيه أي شي خاص بـ DVC.

</div>

The second stage: train a model and write a metrics file. Note it reads `data/prepared.csv` —
the output of stage one. That overlap is how DVC works out the order for itself.

<!-- ar -->
<div dir="rtl" lang="ar">

المرحلة الثانية: درّب نموذجاً واكتب ملف مقاييس. لاحظ أنها تقرأ الملفات المحضّرة، وهي مخرجات المرحلة الأولى. هذا التداخل هو كيف يستنتج DVC الترتيب بنفسه.

</div>

In [21]:
%%writefile src/train.py
"""Stage 2: train a random forest, save the model and a metrics file."""
import pandas as pd, yaml, json, os, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

p = yaml.safe_load(open("params.yaml"))

train = pd.read_csv("data/prepared/train.csv")
test  = pd.read_csv("data/prepared/test.csv")
FEATURES = ["tenure_months", "monthly_charge", "support_calls", "is_premium"]

model = RandomForestClassifier(n_estimators=p["n_estimators"],
                               max_depth=p["max_depth"],
                               random_state=p["random_state"])
model.fit(train[FEATURES], train.churn)

pred = model.predict(test[FEATURES])
prob = model.predict_proba(test[FEATURES])[:, 1]
metrics = {"accuracy": round(accuracy_score(test.churn, pred), 4),
           "f1":       round(f1_score(test.churn, pred), 4),
           "roc_auc":  round(roc_auc_score(test.churn, prob), 4)}

os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/model.pkl")
json.dump(metrics, open("metrics.json", "w"), indent=2)
print("train:", metrics)

Writing src/train.py


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · سكربت المرحلة التانية**

`%%writefile` بتكتب `src/train.py`: بيقرأ المعاملات وملفات التدريب والاختبار المحضّرة، وبيدرّب غابة عشوائية، وبيحسب الدقة و F1 و ROC AUC. بعدين بيحفظ الموديل بـ `models/model.pkl` والمقاييس بـ `metrics.json`.

</div>

## Step 2.3 · Wire them into `dvc.yaml`

Read the `deps`/`outs` carefully: `prepare` **produces** `data/prepared`, and `train`
**depends on** it. That shared path is what makes DVC infer the order — you never write the
order down yourself.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٢٫٣ · اربطهما في `dvc.yaml`**

اقرأ `deps`/`outs` بانتباه: `prepare` **ينتج** `data/prepared`، و `train` **يعتمد على** نفس المسار. هذا المسار المشترك هو ما يجعل DVC يستنتج الترتيب، ولا تكتب الترتيب بنفسك أبداً.

</div>

In [22]:
%%writefile dvc.yaml
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
      - src/prepare.py
      - data/raw.csv          # our dvc-tracked dataset
    params:
      - test_size
      - random_state
    outs:
      - data/prepared         # DVC tracks this folder for us

  train:
    cmd: python src/train.py
    deps:
      - src/train.py
      - data/prepared         # output of `prepare` -> input here. This builds the DAG.
    params:
      - n_estimators
      - max_depth
      - random_state
    outs:
      - models/model.pkl
    metrics:
      - metrics.json:
          cache: false        # small + JSON -> keep it in Git so we can diff it in PRs

Writing dvc.yaml


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · وصف الخط بـ `dvc.yaml`**

`%%writefile` بتكتب `dvc.yaml` بمرحلتين. `prepare` بتعتمد على سكربتها والبيانات الخام ومعاملين، وبتطلع `data/prepared`. و `train` بتعتمد على سكربتها و `data/prepared` نفسه، وهاد يلي بيربط المرحلتين، وبتطلع الموديل، والمقاييس بـ `metrics.json` بدون تخزين مؤقت عشان تنحفظ بـ git ونقدر نقارنها.

</div>

`dvc dag` draws the pipeline. Nobody wrote this graph — DVC derived it from what each stage
declares it reads and writes.

<!-- ar -->
<div dir="rtl" lang="ar">

`dvc dag` يرسم الخط. لم يكتب أحد هذا الرسم، DVC استنتجه مما تعلن كل مرحلة أنها تقرأه وتكتبه.

</div>

In [23]:
!dvc dag        # DVC works out the order from deps/outs and draws it

+------------------+ 
| data/raw.csv.dvc | 
+------------------+ 
          *          
          *          
          *          
    +---------+      
    | prepare |      
    +---------+      
          *          
          *          
          *          
      +-------+      
      | train |      
      +-------+      


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · رسم الخط**

بنشغّل `dvc dag` يلي بيرسم المراحل وترتيبها بالنص، مستنتج من المدخلات والمخرجات.

</div>

## Step 2.4 · `dvc repro` — run it

First run: everything is stale, so both stages execute.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٢٫٤ · `dvc repro`: شغّله**

التشغيل الأول: كل شيء قديم، فتعمل المرحلتان.

</div>

In [24]:
!dvc repro

'data/raw.csv.dvc' didn't change, skipping                                      
Running stage 'prepare':                                                        
> python src/prepare.py
prepare: 2000 train / 500 test rows
  0% Committing data/prepared to cache|              |0/3 [00:00<?,     ?file/s]
!
  0%|          |memory://.zVhjwn_u6ZBrTynNSSAefg.0.00/137 [00:00<?,        ?B/s]
Generating lock file 'dvc.lock'                                                 
Updating lock file 'dvc.lock'

Running stage 'train':                                                          
> python src/train.py
train: {'accuracy': 0.8, 'f1': 0.7942, 'roc_auc': 0.8868}
Updating lock file 'dvc.lock'                                                   

To track the changes with git, run:

	git add dvc.lock data/.gitignore models/.gitignore

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · أول تشغيل للخط**

بنشغّل `dvc repro`، ولأنه أول مرة، المرحلتين بيشتغلوا ورا بعض.

</div>

Now run it **again without changing anything**. Nothing should execute — DVC compares
hashes of every dep and param and skips what is unchanged.

<!-- ar -->
<div dir="rtl" lang="ar">

الآن شغّله **مرة أخرى دون تغيير أي شيء**. لا يجب أن يعمل شيء، لأن DVC يقارن بصمات كل المدخلات والمعاملات ويتجاوز ما لم يتغير.

</div>

In [25]:
!dvc repro

'data/raw.csv.dvc' didn't change, skipping                                      
Stage 'prepare' didn't change, skipping                                         
Stage 'train' didn't change, skipping                                           
Data and pipelines are up to date.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تشغيل تاني بدون تغيير**

بنشغّل `dvc repro` مرة تانية بدون ما نغير شي، و DVC بيقول إنه ما في داعي يشغّل أي مرحلة.

</div>

### `dvc.lock` — the receipt

`dvc repro` wrote a lock file recording the exact hash of every input and output. This is
the machine-readable answer to *"which data and which parameters produced this model?"*.
**Commit it.**

<!-- ar -->
<div dir="rtl" lang="ar">

**`dvc.lock`: الإيصال**

`dvc repro` كتب ملف قفل يسجّل البصمة الدقيقة لكل مدخل ومخرج. هذا هو الجواب المقروء آلياً لسؤال *"أي بيانات وأي معاملات أنتجت هذا النموذج؟"*. **احفظه في git.**

</div>

In [26]:
!cat dvc.lock

schema: '2.0'
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
    - path: data/raw.csv
      hash: md5
      md5: 84846d31147d472492ebb4ecbb74aaa2
      size: 37065
    - path: src/prepare.py
      hash: md5
      md5: 1ba24117e97700ca4e3e0e67a962e64f
      size: 605
    params:
      params.yaml:
        random_state: 42
        test_size: 0.2
    outs:
    - path: data/prepared
      hash: md5
      md5: 3892bc22d1b0d03d426a35e7a875012f.dir
      size: 37125
      nfiles: 2
  train:
    cmd: python src/train.py
    deps:
    - path: data/prepared
      hash: md5
      md5: 3892bc22d1b0d03d426a35e7a875012f.dir
      size: 37125
      nfiles: 2
    - path: src/train.py
      hash: md5
      md5: 20dfac89a04ac08e985911bad4d62201
      size: 1113
    params:
      params.yaml:
        max_depth: 4
        n_estimators: 60
        random_state: 42
    outs:
    - path: metrics.json
      hash: md5
      md5: 743ccdc67afb6015c7f672d015e26188
      size: 58
    - path: models/mo

<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · عرض ملف القفل**

بنعرض محتوى `dvc.lock` بـ `cat`، وفيه بصمة كل مدخل ومخرج وقيم المعاملات لكل مرحلة.

</div>

## Step 2.5 · Change one parameter and watch what re-runs

Bump `n_estimators` from 60 to 150. `prepare` does not depend on it, so DVC must skip
`prepare` and re-run only `train`.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٢٫٥ · غيّر معاملاً واحداً وشاهد ما يُعاد**

غيّر `n_estimators` من ٦٠ إلى ١٥٠. مرحلة `prepare` لا تعتمد عليه، فيجب أن يتجاوزها DVC ويعيد `train` فقط.

</div>

In [27]:
cfg = open("params.yaml").read().replace("n_estimators: 60", "n_estimators: 150")
open("params.yaml", "w").write(cfg)

!dvc repro      # look closely: 'prepare' is skipped, 'train' runs

!dvc metrics show

'data/raw.csv.dvc' didn't change, skipping                                      
Stage 'prepare' didn't change, skipping                                         
Running stage 'train':                                                          
> python src/train.py
train: {'accuracy': 0.808, 'f1': 0.8008, 'roc_auc': 0.8889}
Updating lock file 'dvc.lock'                                                   

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.
Path          accuracy    f1      roc_auc
metrics.json  0.808       0.8008  0.8889


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تغيير معامل واحد**

بنقرأ `params.yaml` وبنغيّر عدد الأشجار من 60 لـ 150 وبنكتبه. بعدين `dvc repro`: لاحظ إنه `prepare` ما اشتغلت لأنها ما بتعتمد على هاد المعامل، و `train` بس اشتغلت. آخر شي بنعرض المقاييس.

</div>

## Step 2.6 · Comparing metrics across commits

`dvc metrics diff` is what turns "the model got better" into a number a reviewer can see.
Let's commit this state, then ask DVC how it compares to the previous commit.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٢٫٦ · مقارنة المقاييس بين الـ commits**

`dvc metrics diff` هو ما يحوّل "النموذج تحسّن" إلى رقم يراه المراجع. لنحفظ هذه الحالة، ثم نسأل DVC كيف تقارن بالـ commit السابق.

</div>

In [28]:
!git add dvc.yaml dvc.lock params.yaml metrics.json src .gitignore
!git commit -q -m "feat: add dvc pipeline (n_estimators=150)"

# tweak again so there is something to diff against.
# Read FIRST, then write -- open(..., "w") empties the file immediately.
cfg = open("params.yaml").read().replace("max_depth: 4", "max_depth: 8")
open("params.yaml", "w").write(cfg)
!dvc repro -q
!dvc metrics diff HEAD

train: {'accuracy': 0.852, 'f1': 0.8452, 'roc_auc': 0.9172}
Path          Metric    HEAD    workspace    Change
metrics.json  accuracy  0.808   0.852        0.044
metrics.json  f1        0.8008  0.8452       0.0444
metrics.json  roc_auc   0.8889  0.9172       0.0283


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · مقارنة المقاييس**

بنحفظ الخط والملفات بـ commit. بعدين بنغيّر العمق من 4 لـ 8 (بنقرأ الملف أول وبعدين بنكتب، لأنه فتحه للكتابة بيفضّيه فوراً)، وبنعيد التشغيل بهدوء، وبنشغّل `dvc metrics diff HEAD` عشان نشوف الفرق بالأرقام بين الحالي والـ commit الأخير.

</div>

In `07-dvc/cml-example.yaml` (shipped with this repo) this exact command runs inside a GitHub
Action and posts the table as a **pull request comment**, alongside `dvc plots diff` charts.
That is CML — model review becomes code review.

<!-- ar -->
<div dir="rtl" lang="ar">

في ملف `cml-example.yaml` الموجود في مجلد DVC بهذا المستودع، يعمل هذا الأمر نفسه داخل GitHub Action وينشر الجدول **كتعليق على pull request**، مع رسوم `dvc plots diff`. هذا هو CML: مراجعة النموذج تصبح مراجعة كود.

</div>

In [29]:
!git add -A && git commit -q -m "exp: max_depth 8"
!git log --oneline

d75d42c (HEAD -> master) exp: max_depth 8
b761ebe feat: add dvc pipeline (n_estimators=150)
8e07377 data: add March rows (v2)
292ddfa chore: add default dvc remote
026e9c2 data: add raw.csv v1
b397cb0 chore: dvc init
1daf1c7 chore: initial commit


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · حفظ التجربة**

بنعمل commit لكل التغييرات، وبنعرض سجل الـ commits.

</div>

---
# Part 3 — DVCLive   ·   deck slides 34–35

## Step 3.1 · Logging from inside your training code

Everything so far has been the CLI. **DVCLive** is the Python half: a few lines in your
training loop and each run becomes a DVC experiment that `dvc exp show` and `dvc plots show`
already understand.

This is the piece people miss — it means you do **not** need a separate tracking service to
compare runs.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الثالث: DVCLive**

**الخطوة ٣٫١ · التسجيل من داخل كود التدريب**

كل ما سبق كان سطر الأوامر. **DVCLive** هو النصف الخاص بـ Python: أسطر قليلة في حلقة التدريب، وكل تشغيل يصبح تجربة DVC يفهمها `dvc exp show` و `dvc plots show` مباشرة.

هذه القطعة التي يفوّتها الناس: تعني أنك **لا** تحتاج خدمة تتبّع منفصلة لمقارنة التشغيلات.

</div>

In [30]:
# commit the pipeline state so DVCLive's experiment snapshot is clean
# commit only if there is something to commit (keeps the output clean either way)
!git add -A && (git diff --cached --quiet || git commit -q -m "chore: before dvclive")
!pip install -q dvclive


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تجهيز وتثبيت DVCLive**

بنحفظ الحالة الحالية بـ commit بس لو في شي جديد، عشان لقطة التجربة تكون نظيفة. بعدين بنثبت `dvclive`.

</div>

**DVCLive** logs metrics from inside your training code, one line at a time, the way a training
loop actually produces them — instead of you assembling a metrics file at the end.

<!-- ar -->
<div dir="rtl" lang="ar">

**DVCLive** يسجّل المقاييس من داخل كود التدريب، سطراً بسطر، بالطريقة التي تنتجها حلقة التدريب فعلاً، بدل أن تجمع ملف مقاييس في النهاية.

</div>

In [31]:
from dvclive import Live
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

train = pd.read_csv("data/prepared/train.csv")
test  = pd.read_csv("data/prepared/test.csv")
FEATURES = ["tenure_months", "monthly_charge", "support_calls", "is_premium"]

# save_dvc_exp=True -> when the block exits, this run is saved as a DVC experiment
with Live(save_dvc_exp=True, exp_name="dvclive-demo") as live:
    live.log_param("max_depth", 6)                 # an input you chose
    live.log_params({"random_state": 42, "logger": "dvclive"})

    # pretend each iteration is an epoch: log a metric per step
    for n in [20, 40, 80, 160]:
        m = RandomForestClassifier(n_estimators=n, max_depth=6,
                                   random_state=42).fit(train[FEATURES], train.churn)
        pred = m.predict(test[FEATURES])
        live.log_metric("roc_auc",  roc_auc_score(test.churn,
                                                  m.predict_proba(test[FEATURES])[:, 1]))
        live.log_metric("accuracy", accuracy_score(test.churn, pred))
        live.next_step()                           # advance the step counter

    live.log_sklearn_plot("confusion_matrix", test.churn.tolist(), pred.tolist())

<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · التسجيل بـ DVCLive**

بنفتح `Live` مع `save_dvc_exp=True` عشان التشغيل ينحفظ كتجربة DVC لما يخلص. بنسجّل الإعدادات بـ `log_param`. بعدين حلقة على 4 أعداد أشجار كأنها دورات تدريب: كل مرة بندرّب وبنسجّل ROC AUC والدقة بـ `log_metric`، وبعدين `next_step` بتقدّم العدّاد. آخر شي بنسجّل رسمة مصفوفة الالتباس.

</div>

Look at what it wrote. Nothing here is a database — it is plain files in the repo.

<!-- ar -->
<div dir="rtl" lang="ar">

انظر إلى ما كتبه. لا شيء هنا قاعدة بيانات، إنها ملفات عادية في المستودع.

</div>

In [32]:
!find dvclive -type f | sort
!cat dvclive/metrics.json          # the latest value of every metric
print("--- one row per step, ready for dvc plots ---")
!head -3 dvclive/plots/metrics/roc_auc.tsv

print("--- DVCLive also registered its plots in dvc.yaml ---")
!grep -A4 "^plots:" dvc.yaml

dvclive/metrics.json
dvclive/params.yaml
dvclive/plots/metrics/accuracy.tsv
dvclive/plots/metrics/roc_auc.tsv
dvclive/plots/sklearn/confusion_matrix.json
{
    "roc_auc": 0.9075923436399719,
    "accuracy": 0.828,
    "step": 3
}
--- one row per step, ready for dvc plots ---
step	roc_auc
0	0.9017668523141924
1	0.9050076819665834
--- DVCLive also registered its plots in dvc.yaml ---
plots:
- dvclive/plots/metrics:
    x: step
- dvclive/plots/sklearn/confusion_matrix.json:
    template: confusion


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · ملفات DVCLive**

بنعرض كل الملفات يلي كتبها DVCLive بمجلد `dvclive`، وبنعرض آخر قيمة لكل مقياس، وأول أسطر من ملف قيم ROC AUC لكل خطوة. وبنشوف كمان إنه DVCLive أضاف قسم `plots:` لملف `dvc.yaml`.

</div>

And the run shows up in the same table as the `dvc exp run` experiments from Step 3.3 —
one place for everything, whether the numbers came from the CLI or from Python.

<!-- ar -->
<div dir="rtl" lang="ar">

والتشغيل يظهر في نفس الجدول مع تجارب `dvc exp run`، مكان واحد لكل شيء، سواء جاءت الأرقام من سطر الأوامر أو من Python.

</div>

In [33]:
!dvc exp show --only-changed

 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  Experiment                   Created    dvclive/metrics.json:roc_auc   dvclive/metrics.json:accuracy   step   dvclive/params.yaml:max_depth   dvclive/params.yaml:random_state   logger   
 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  workspace                    -                               0.90759                           0.828      3   6                               42                                 dvclive  
  master                       12:16 PM                              -                               -      -   -                               -                                  -        
  └── 2ef6682 [dvclive-demo]   12:16 PM                

<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · جدول التجارب**

بنشغّل `dvc exp show --only-changed` عشان نعرض جدول التجارب مع الأعمدة يلي تغيرت بس.

</div>

> **Reference:** deck slide 35 lists every `Live()` argument (`monitor_system`, `report`,
> `resume`, …) and the framework callbacks for Keras, Lightning, HuggingFace, XGBoost and more.

---
# Part 4 — Advanced   ·   deck slides 41–54

The 10% you will want in month two. Every step here runs in the sandbox you just built.

## Step 4.1 · Read any version of the data straight from Python

`dvc.api` reads a tracked file at **any git revision** without checking anything out — how a
downstream script pins itself to a known dataset version.

<!-- ar -->
<div dir="rtl" lang="ar">

> **للمراجعة:** العرض يسرد كل خيارات `Live()` (`monitor_system`، و `report`، و `resume`، …) وأدوات الربط مع Keras و Lightning و HuggingFace و XGBoost وغيرها.

**الجزء الرابع: متقدم**

الـ ١٠٪ التي ستريدها في الشهر الثاني. كل خطوة هنا تعمل في مجلد التجربة الذي بنيته للتو.

**الخطوة ٤٫١ · اقرأ أي نسخة من البيانات مباشرة من Python**

`dvc.api` يقرأ ملفاً متتبَّعاً في **أي نسخة git** دون الانتقال إليها، وهكذا يثبّت سكربت لاحق نفسه على نسخة بيانات معروفة.

</div>

In [34]:
import dvc.api, pandas as pd, io

# find the commit where raw.csv was FIRST tracked -- that is the v1 dataset
hist = !git log --format=%H -- data/raw.csv.dvc
V1 = hist[-1]                      # git log is newest-first, so the last entry is oldest

# read the current version
cur = pd.read_csv(io.StringIO(dvc.api.read("data/raw.csv", repo=".")))

# read the v1 version -- no checkout, no cd, nothing moved on disk
old = pd.read_csv(io.StringIO(dvc.api.read("data/raw.csv", repo=".", rev=V1)))

print("current  :", len(cur), "rows")
print(f"at {V1[:8]}:", len(old), "rows   <- the pre-March dataset")
print("\nand the working folder was never touched:", len(pd.read_csv("data/raw.csv")), "rows on disk")


current  : 2500 rows
at 5bf2524c: 2000 rows   <- the pre-March dataset

and the working folder was never touched: 2500 rows on disk


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · قراءة نسخة قديمة من بايثون**

بنجيب من سجل git أول commit انضاف فيه ملف المؤشر، وهاد هو v1. بعدين بـ `dvc.api.read` بنقرأ النسخة الحالية ونسخة v1 مباشرة كنص، وبنحولهم لجداول. بنطبع عدد الصفوف بكل نسخة، وبنتأكد إنه الملف على القرص ما تغير أبداً.

</div>

## Step 4.2 · What changed, and where does it live?

Three inspection commands you will reach for constantly once a project is real.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٤٫٢ · ما الذي تغيّر، وأين يوجد؟**

ثلاثة أوامر فحص ستستخدمها باستمرار عندما يصبح المشروع حقيقياً.

</div>

In [35]:
!dvc diff HEAD~1 HEAD      # which tracked files changed between two commits

!dvc data status           # modified / not in cache / not in remote, in one view
!dvc status -c             # specifically: am I in sync with the remote?

 ⠋ Calculating diff
Modified:
    models/model.pkl

files summary: 1 modified
Calculating diff between index/workspace              |0.00 [00:00,    ?entry/s]
!
  0%|          |Checking cache                        0/3 [00:00<?,    ?entry/s]
No changes.                                                                     
(there are changes not tracked by dvc, use "git status" to see)
	new:                data/prepared/train.csv                                    
	new:                data/prepared/test.csv
	new:                data/prepared
	new:                models/model.pkl


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · أوامر الفحص**

`dvc diff` بيعرض الملفات المتتبعة يلي تغيرت بين commit وآخر. و `dvc data status` بيعرض بنظرة وحدة شو معدّل وشو مش بالذاكرة المؤقتة وشو مش بالمخزن البعيد. و `dvc status -c` بيقارن تحديداً مع المخزن البعيد.

</div>

## Step 4.3 · Experiments without polluting Git history

`dvc exp run` runs the pipeline with overridden params and stores the result as a **hidden
commit**. No branch clutter — you promote only the winner.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٤٫٣ · تجارب دون تلويث تاريخ Git**

`dvc exp run` يشغّل الخط بمعاملات مُعدّلة ويحفظ النتيجة كـ **commit مخفي**. لا فروع مزدحمة، وترقّي الفائز فقط.

</div>

In [36]:
# -S overrides a param for this run only; params.yaml on disk is left alone
!dvc exp run -S n_estimators=40  --name small
!dvc exp run -S n_estimators=250 --name big

!dvc exp show --only-changed        # the table of every attempt, with metrics

Reproducing experiment 'small'                                                  
Building workspace index                             |8.00 [00:00, 1.56kentry/s]
Comparing indexes                                    |8.00 [00:00, 4.25kentry/s]
Applying changes                                      |0.00 [00:00,     ?file/s]
'data/raw.csv.dvc' didn't change, skipping                                      
Stage 'prepare' didn't change, skipping                                         
Running stage 'train':                                                          
> python src/train.py
train: {'accuracy': 0.846, 'f1': 0.8386, 'roc_auc': 0.914}
Updating lock file 'dvc.lock'                                                   
 
Ran experiment(s): small
Experiment results have been applied to your workspace.
Reproducing experiment 'big'                                                    
Building workspace index                             |8.00 [00:00, 1.63kentry/s]
Comparing indexes         

<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تجارب بمعاملات مختلفة**

بنشغّل تجربتين بـ `dvc exp run -S`: وحدة بـ 40 شجرة ووحدة بـ 250، وكل وحدة إلها اسم. الخيار `-S` بيغير المعامل لهاد التشغيل بس، والملف على القرص ما بيتغير. بعدين بنعرض جدول كل التجارب مع مقاييسها.

</div>

> **Reference:** deck slide 44 covers running these at scale — `dvc exp run --queue`,
> `dvc queue start -j 4`, `dvc exp branch` to promote a winner.

## Step 4.4 · Templated stages — stop copy-pasting YAML

`foreach` generates one stage per item. `matrix` generates the cartesian product. Both keep
the "only re-run what went stale" behaviour per generated stage.

<!-- ar -->
<div dir="rtl" lang="ar">

> **للمراجعة:** العرض يغطي تشغيل هذه التجارب على نطاق واسع: `dvc exp run --queue`، و `dvc queue start -j 4`، و `dvc exp branch` لترقية الفائز.

**الخطوة ٤٫٤ · مراحل بقوالب، توقف عن نسخ YAML ولصقه**

`foreach` ينشئ مرحلة لكل عنصر. و `matrix` ينشئ كل التوافيق. والاثنان يحافظان على سلوك "أعد تشغيل القديم فقط" لكل مرحلة مولّدة.

</div>

In [37]:
%%writefile src/slice.py
"""Write a per-segment CSV so we have something for `foreach` to iterate over."""
import sys, pandas as pd, os

segment = sys.argv[1]                       # "premium" or "standard"
df = pd.read_csv("data/prepared/train.csv")
sub = df[df.is_premium == (1 if segment == "premium" else 0)]

os.makedirs("data/segments", exist_ok=True)
sub.to_csv(f"data/segments/{segment}.csv", index=False)
print(f"{segment}: {len(sub)} rows")

Writing src/slice.py


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · سكربت لكل شريحة**

`%%writefile` بتكتب `src/slice.py`: بياخد اسم الشريحة (premium أو standard) من سطر الأوامر، وبيفلتر بيانات التدريب حسبها، وبيحفظها بملف خاص. هاد عشان يكون عنا شي نكرره بـ `foreach`.

</div>

A **templated stage**: one description that DVC expands into several stages, one per item in the
list. Useful when you train the same thing per segment, per region, or per language.

<!-- ar -->
<div dir="rtl" lang="ar">

**مرحلة بقالب**: وصف واحد يوسّعه DVC إلى عدة مراحل، واحدة لكل عنصر في القائمة. مفيد عندما تدرّب نفس الشيء لكل شريحة، أو منطقة، أو لغة.

</div>

In [38]:
# `dvc.yaml` is just YAML, so we add the templated stage under `stages:` properly.
# (Appending text blindly would be fragile -- DVCLive already added a `plots:` section
#  in Part 3, and an appended block would land under the wrong parent key.)
import yaml

cfg = yaml.safe_load(open("dvc.yaml"))
cfg["stages"]["slice"] = {
    "foreach": ["premium", "standard"],          # one generated stage per entry
    "do": {
        "cmd":  "python src/slice.py ${item}",
        "deps": ["src/slice.py", "data/prepared"],
        "outs": ["data/segments/${item}.csv"],
    },
}
yaml.safe_dump(cfg, open("dvc.yaml", "w"), sort_keys=False)

print(open("dvc.yaml").read())

stages:
  prepare:
    cmd: python src/prepare.py
    deps:
    - src/prepare.py
    - data/raw.csv
    params:
    - test_size
    - random_state
    outs:
    - data/prepared
  train:
    cmd: python src/train.py
    deps:
    - src/train.py
    - data/prepared
    params:
    - n_estimators
    - max_depth
    - random_state
    outs:
    - models/model.pkl
    metrics:
    - metrics.json:
        cache: false
  slice:
    foreach:
    - premium
    - standard
    do:
      cmd: python src/slice.py ${item}
      deps:
      - src/slice.py
      - data/prepared
      outs:
      - data/segments/${item}.csv
params:
- dvclive/params.yaml
metrics:
- dvclive/metrics.json
plots:
- dvclive/plots/metrics:
    x: step
- dvclive/plots/sklearn/confusion_matrix.json:
    template: confusion
    x: actual
    y: predicted
    title: Confusion Matrix
    x_label: True Label
    y_label: Predicted Label



<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · إضافة مرحلة `foreach`**

بدل ما نضيف نص على آخر `dvc.yaml` (وهاد خطر لأنه DVCLive أضاف قسم `plots:`)، بنقرأ الملف كـ YAML، وبنضيف مرحلة `slice` فيها `foreach` على الشريحتين، والأمر والمدخلات والمخرجات فيهم `${item}`. بعدين بنكتب الملف وبنطبعه.

</div>

Now `dvc repro` runs each generated stage, and the graph branches.

<!-- ar -->
<div dir="rtl" lang="ar">

الآن `dvc repro` يشغّل كل مرحلة مولّدة، والرسم يتفرّع.

</div>

In [39]:
!dvc repro          # note the generated stage names: slice@premium, slice@standard

!dvc dag            # the graph now branches

'data/raw.csv.dvc' didn't change, skipping                                      
Stage 'prepare' didn't change, skipping                                         
Stage 'train' didn't change, skipping                                           
Running stage 'slice@premium':                                                  
> python src/slice.py premium
premium: 1019 rows
Updating lock file 'dvc.lock'                                                   

Running stage 'slice@standard':                                                 
> python src/slice.py standard
standard: 981 rows
Updating lock file 'dvc.lock'                                                   

To track the changes with git, run:

	git add dvc.lock data/segments/.gitignore

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.
                   +------------------+                         
                   | data/raw.csv.dvc |                         
    

<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تشغيل المراحل المولّدة**

بنشغّل `dvc repro`، ولاحظ أسماء المراحل الجديدة `slice@premium` و `slice@standard`. بعدين `dvc dag` بيبين إنه الرسم صار متفرع.

</div>

`matrix` is the same idea across two axes — six stages from four lines:

```yaml
  train:
    matrix:
      model: [rf, xgb]
      split: [a, b, c]
    cmd: python src/train.py --model ${item.model} --split ${item.split}
    outs: [models/${item.model}-${item.split}.pkl]
```

Values can come from `params.yaml` too (`foreach: ${cities}`), and `dvc repro -j 4` runs
independent stages in parallel.


## Step 4.5 · The run cache — why a stage sometimes doesn't run

DVC records a signature of every stage execution in `.dvc/cache/runs`. If it has seen this
exact combination of command, dependencies and params before, it **restores the outputs**
instead of recomputing them.

Watch it happen: change `max_depth` to a value never used, run it, then change it straight
back to a value we *have* just used.

<!-- ar -->
<div dir="rtl" lang="ar">

`matrix` نفس الفكرة على محورين: ست مراحل من أربعة أسطر، مثل نموذجين × ثلاثة تقسيمات كما في المثال أعلاه.

القيم يمكن أن تأتي من `params.yaml` أيضاً (`foreach: ${cities}`)، و `dvc repro -j 4` يشغّل المراحل المستقلة بالتوازي.

**الخطوة ٤٫٥ · ذاكرة التشغيل: لماذا لا تعمل مرحلة أحياناً**

DVC يسجّل توقيع كل تنفيذ مرحلة في `.dvc/cache/runs`. إذا رأى هذا المزيج بالضبط من الأمر والاعتماديات والمعاملات من قبل، **يستعيد المخرجات** بدل حسابها من جديد.

شاهده يحدث: غيّر `max_depth` إلى قيمة لم تُستخدم، وشغّل، ثم أرجعها مباشرة إلى قيمة استخدمناها للتو.

</div>

In [40]:
# A -> B : max_depth 3 has never been run, so this one really trains
cfg = open("params.yaml").read().replace("max_depth: 8", "max_depth: 3")
open("params.yaml", "w").write(cfg)

!dvc repro train


'data/raw.csv.dvc' didn't change, skipping                                      
Stage 'prepare' didn't change, skipping                                         
Running stage 'train':                                                          
> python src/train.py
train: {'accuracy': 0.794, 'f1': 0.7894, 'roc_auc': 0.8708}
Updating lock file 'dvc.lock'                                                   

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تشغيل بقيمة جديدة**

بنغير العمق من 8 لـ 3، وهي قيمة ما جربناها قبل، وبنشغّل مرحلة `train` بس. هون التدريب بيصير فعلاً.

</div>

That one executed. Now go straight back to `max_depth: 8` — the exact configuration DVC
ran moments ago, with the same dependencies.

<!-- ar -->
<div dir="rtl" lang="ar">

هذه عملت فعلاً. الآن ارجع مباشرة إلى `max_depth: 8`، وهو الإعداد نفسه الذي شغّله DVC قبل لحظات، بنفس الاعتماديات.

</div>

In [41]:
# B -> A : this signature is already in .dvc/cache/runs
cfg = open("params.yaml").read().replace("max_depth: 3", "max_depth: 8")
open("params.yaml", "w").write(cfg)

!dvc repro train


'data/raw.csv.dvc' didn't change, skipping                                      
Stage 'prepare' didn't change, skipping                                         
Stage 'train' is cached - skipping run, checking out outputs                    
Updating lock file 'dvc.lock'                                                   

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · الرجوع لقيمة قديمة**

بنرجع العمق لـ 8 وبنشغّل `train` من جديد. هاد التوقيع موجود بذاكرة التشغيل، فـ DVC بيجيب المخرجات منها بدل ما يدرّب.

</div>

Compare the two outputs:

```
Running stage 'train':                                  <- the first one really trained
Stage 'train' is cached - skipping run, checking out outputs   <- the second one did not
```

DVC recognised the signature and took the outputs straight from `.dvc/cache/runs`. That cache
travels with `dvc push` / `dvc pull`, so a colleague who tries a configuration you already ran
gets your results without spending the compute again.

```bash
dvc repro --no-run-cache    # force real execution
dvc repro -f                # force every stage
dvc repro -s train          # a single stage
```

<!-- ar -->
<div dir="rtl" lang="ar">

قارن المخرجين: الأول درّب فعلاً (`Running stage 'train'`)، والثاني لم يدرّب (`Stage 'train' is cached - skipping run`).

DVC تعرّف على التوقيع وأخذ المخرجات مباشرة من `.dvc/cache/runs`. هذه الذاكرة تنتقل مع `dvc push` / `dvc pull`، فالزميل الذي يجرب إعداداً شغّلته أنت يحصل على نتائجك دون أن يصرف الحساب مرة أخرى.

للإجبار: `--no-run-cache` يفرض تنفيذاً حقيقياً، و `-f` يفرض كل المراحل، و `-s train` مرحلة واحدة.

</div>

In [42]:
# make sure the workspace and the lock file agree before moving on
!dvc repro -q
!git add -A && git commit -q -m "chore: params back to max_depth 8"


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · مطابقة الحالة**

بنشغّل `dvc repro` بهدوء عشان الملفات وملف القفل يتطابقوا، وبنحفظ كل شي بـ commit قبل ما نكمل.

</div>

## Step 4.6 · `.dvcignore`

Same syntax as `.gitignore`, but it tells **DVC** what to skip when it scans a tracked
directory. Useful when a data folder is full of scratch files you do not want hashed.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٤٫٦ · `.dvcignore`**

نفس صيغة `.gitignore`، لكنه يخبر **DVC** بما يتجاوزه عندما يفحص مجلداً متتبَّعاً. مفيد عندما يكون مجلد البيانات مليئاً بملفات مؤقتة لا تريد حساب بصمتها.

</div>

In [43]:
%%writefile .dvcignore
# scratch files inside tracked directories -- never hash these
*.tmp
.ipynb_checkpoints/
**/__pycache__/

Overwriting .dvcignore


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · ملف `.dvcignore`**

`%%writefile` بتكتب `.dvcignore` بيستثني الملفات المؤقتة `*.tmp`، ونقاط حفظ Jupyter، ومجلدات `__pycache__`، عشان DVC ما يحسب بصمتها.

</div>

`.dvcignore` works like `.gitignore`, but for DVC: it keeps scratch files out of the hashing, so
a temporary file cannot make a stage look stale.

<!-- ar -->
<div dir="rtl" lang="ar">

`.dvcignore` يعمل مثل `.gitignore` لكن لـ DVC: يُبقي الملفات المؤقتة خارج حساب البصمات، فلا يستطيع ملف مؤقت أن يجعل مرحلة تبدو قديمة.

</div>

In [44]:
!git add .dvcignore && git commit -q -m "chore: add .dvcignore"
!dvc status          # unchanged: the ignore file only affects what DVC scans

Data and pipelines are up to date.                                              


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · حفظ ملف التجاهل**

بنحفظ `.dvcignore` بـ commit، وبنشغّل `dvc status` عشان نشوف إنه ما في شي تغير، لأنه ملف التجاهل بيأثر بس على شو DVC بيفحص.

</div>

## Step 4.7 · `artifacts:` — DVC's own model registry

Add an `artifacts:` block to `dvc.yaml` and your model gets a **name**, a type and labels.
Consumers then fetch it by name and git revision instead of guessing at paths.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٤٫٧ · `artifacts:`: سجل النماذج الخاص بـ DVC**

أضف كتلة `artifacts:` إلى `dvc.yaml`، فيحصل نموذجك على **اسم**، ونوع، وتصنيفات. المستخدمون يجلبونه بالاسم ونسخة git بدل تخمين المسارات.

</div>

In [45]:
import yaml

cfg = yaml.safe_load(open("dvc.yaml"))
cfg["artifacts"] = {
    "churn-classifier": {                        # the artifact ID
        "path": "models/model.pkl",              # the only required field
        "type": "model",                         # "model" is what the registry lists
        "desc": "Churn random forest, sklearn",
        "labels": ["tabular", "sklearn"],
        "meta": {"framework": "scikit-learn"},
    }
}
yaml.safe_dump(cfg, open("dvc.yaml", "w"), sort_keys=False)

print(yaml.safe_load(open("dvc.yaml"))["artifacts"])

{'churn-classifier': {'path': 'models/model.pkl', 'type': 'model', 'desc': 'Churn random forest, sklearn', 'labels': ['tabular', 'sklearn'], 'meta': {'framework': 'scikit-learn'}}}


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تعريف الموديل كـ artifact**

بنقرأ `dvc.yaml` كـ YAML، وبنضيف قسم `artifacts` فيه `churn-classifier` مع مسار الموديل (وهو الحقل الوحيد المطلوب)، ونوعه model، ووصف، وتصنيفات، ومعلومات إضافية. بعدين بنكتب الملف وبنطبع القسم الجديد.

</div>

An **artifact** entry names a model in `dvc.yaml` so other repositories can fetch it by name
instead of by path. The versions come from git tags.

<!-- ar -->
<div dir="rtl" lang="ar">

إدخال **artifact** يسمّي نموذجاً في `dvc.yaml` حتى تجلبه المستودعات الأخرى بالاسم بدل المسار. والإصدارات تأتي من وسوم git.

</div>

In [46]:
!git add dvc.yaml && git commit -q -m "feat: declare model artifact"

# DVC reads artifact VERSIONS from git tags shaped  <artifact-name>@<version>
!git tag churn-classifier@v1.0.0
!git tag -l


churn-classifier@v1.0.0


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · إصدار الموديل بوسم git**

بنحفظ `dvc.yaml` بـ commit، وبعدين بنعمل وسم git بالشكل `churn-classifier@v1.0.0`، لأنه DVC بيقرأ إصدارات الـ artifact من وسوم بهاد الشكل. بنعرض الوسوم.

</div>

---
# Part 5 — The real thing: `data.zip`   ·   deck slides 5–8

Everything so far used a small generated CSV so the notebook runs anywhere. Now do it once on
the **actual dataset from the deck** — `data.zip`, 39 MB of cat and dog photographs, which
ships in this repository next to the notebook.

This is the exact scenario slides 5–6 describe: a big zip sitting loose in a folder, no
version, no history. Watch it become a tracked dataset that Git carries in five lines of text.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الخامس: الشيء الحقيقي: `data.zip`**

كل ما سبق استخدم ملف CSV صغيراً مولّداً حتى يعمل الدفتر في أي مكان. الآن افعلها مرة على **البيانات الحقيقية من العرض**: `data.zip`، ٣٩ ميغابايت من صور القطط والكلاب، موجودة في هذا المستودع بجانب الدفتر.

هذا بالضبط السيناريو الذي تصفه الشرائح الأولى: ملف مضغوط كبير متروك في مجلد، بلا نسخة، وبلا تاريخ. شاهده يصبح بيانات متتبَّعة يحملها Git في خمسة أسطر نصية.

</div>

In [47]:
import pathlib, zipfile

# data.zip ships with this repo, right next to the notebook. The other paths are fallbacks
# in case you moved it, or are running this notebook outside the repo.
CANDIDATES = [BASE / "data.zip", BASE.parent / "data.zip", BASE.parent.parent / "data.zip",
              pathlib.Path.home() / "data.zip"]
DATA_ZIP = next((p for p in CANDIDATES if p.is_file()), None)

if DATA_ZIP:
    z = zipfile.ZipFile(DATA_ZIP)
    files = [n for n in z.namelist() if not n.endswith("/")]
    print(f"found {DATA_ZIP}")
    print(f"  {DATA_ZIP.stat().st_size/1024/1024:.1f} MB on disk, {len(files)} files inside")
    print("  e.g.", files[1])
else:
    print("data.zip not found — Part 5 will skip itself. Looked in:")
    for p in CANDIDATES: print("   ", p)

found /home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-free-traning/dvc/data.zip
  39.2 MB on disk, 1800 files inside
  e.g. data/train/dogs/dog.205.jpg


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · إيجاد `data.zip`**

بندور على `data.zip` بكذا مكان: جنب الدفتر، وبالمجلد الأب، وبمجلد المستخدم. لو لقيناه بنفتحه بـ `zipfile` وبنطبع حجمه وعدد الملفات ومثال. ولو ما لقيناه بنطبع الأماكن يلي دورنا فيها، والجزء الخامس بيتجاوز نفسه.

</div>

## Step 5.1 · Unpack it and track it

`dvc add` on a **folder** behaves exactly like `dvc add` on a file: one pointer for the whole
tree, with a count of how many files are inside.

We unpack into `photos/` rather than `data/`, because this sandbox already uses `data/` for the
pipeline's outputs — and DVC refuses to track a folder that overlaps something it already
tracks. (That error message is a useful one to have seen once.)

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٥٫١ · فك الضغط والتتبّع**

`dvc add` على **مجلد** يتصرف تماماً مثل `dvc add` على ملف: مؤشر واحد للشجرة كلها، مع عدد الملفات داخلها.

نفك الضغط إلى `photos/` بدل `data/`، لأن مجلد التجربة يستخدم `data/` لمخرجات الخط، و DVC يرفض تتبّع مجلد يتداخل مع شيء يتتبعه أصلاً. (رسالة الخطأ هذه مفيدة أن تراها مرة.)

</div>

In [48]:
if DATA_ZIP:
    import os, shutil
    # The zip contains a top-level `data/`, but this sandbox already uses `data/` for the
    # pipeline's own tracked outputs -- DVC refuses to track a folder that overlaps them.
    # So unpack the photos into their own directory.
    shutil.rmtree("_unzip", ignore_errors=True); shutil.rmtree("photos", ignore_errors=True)
    z.extractall("_unzip")
    os.rename("_unzip/data", "photos")
    shutil.rmtree("_unzip", ignore_errors=True)

    !du -sh photos/train photos/validation
    print("jpg files:", len(list(pathlib.Path("photos").rglob("*.jpg"))))

24M	photos/train
20M	photos/validation
jpg files: 1800


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · فك ضغط الصور**

لو الملف موجود، بنفك الضغط بمجلد مؤقت، وبننقل مجلد `data` يلي جواه لاسم `photos`، لأنه `data` مستخدم للخط و DVC ما بيقبل تداخل. بعدين بنطبع حجم مجلدي التدريب والتحقق وعدد صور jpg.

</div>

This part uses the real photo folder shipped with the repository — 1,800 files. Tracking a
*directory* is where DVC's advantage over git becomes obvious.

<!-- ar -->
<div dir="rtl" lang="ar">

هذا الجزء يستخدم مجلد الصور الحقيقي الموجود مع المستودع، ١٨٠٠ ملف. تتبّع *مجلد* هو المكان الذي تصبح فيه ميزة DVC على git واضحة.

</div>

In [49]:
if DATA_ZIP:
    !dvc add photos        # note: a directory, not a single file

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-f
                                                                                
!
  0%|          |Adding photos to cache         0.00/1.80k [00:00<?,     ?file/s]
100%|█████████▉|Adding photos to cache    1.80k/1.80k [00:00<00:00, 18.0kfile/s]
                                                                                
!
Checking out /home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-free-traning
100% Adding...|████████████████████████████████████████|1/1 [00:00,  3.67file/s]

To track the changes with git, run:

	git add .gitignore photos.dvc

To enable auto staging, run:

	dvc config core.autostage true


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · `dvc add` على مجلد**

بنشغّل `dvc add photos` على المجلد كامل، مش ملف واحد.

</div>

## Step 5.2 · What Git is now carrying

The whole photo library is represented by one small text file. Compare the two sizes — this is
the slide-5 claim, reproduced on your own machine.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٥٫٢ · ماذا يحمل Git الآن**

مكتبة الصور كلها ممثّلة بملف نصي صغير واحد. قارن الحجمين، هذا ادعاء الشريحة الخامسة، مُعاداً على جهازك.

</div>

In [50]:
if DATA_ZIP:
    print("--- the pointer Git will store ---")
    !cat photos.dvc
    print("--- sizes ---")
    !du -sh photos                     # the photographs
    !du -sh --apparent-size .git       # everything Git is tracking
    !git add photos.dvc .gitignore && git commit -q -m "data: add cats & dogs photo set"
    !git log --oneline -1

--- the pointer Git will store ---
outs:
- md5: b8f4d5a78e55e88906d5f4aeaf43802e.dir
  size: 41149064
  nfiles: 1800
  hash: md5
  path: photos
--- sizes ---
43M	photos
61K	.git
a37d810 (HEAD -> master) data: add cats & dogs photo set


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · مقارنة الأحجام**

بنعرض ملف المؤشر `photos.dvc`، وبنقارن حجم مجلد الصور بحجم كل شي git بيحفظه. بعدين بنحفظ المؤشر وملف التجاهل بـ commit وبنعرضه.

</div>

## Step 5.3 · Push it, delete it, get it back

The same three commands from Part 1 — now on 39 MB of images instead of a toy CSV.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٥٫٣ · ارفعها، واحذفها، واسترجعها**

نفس الأوامر الثلاثة من الجزء الأول، الآن على ٣٩ ميغابايت من الصور بدل ملف CSV صغير.

</div>

In [51]:
if DATA_ZIP:
    !dvc push
    print("\n--- the remote now holds the photographs ---")
    !du -sh "{REMOTE}"

Pushing
!
  0% Querying remote cache|                          |0/2 [00:00<?,    ?files/s]
                                                                                
!
  0% Checking cache in '/home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-f
                                                                                
!
  0% Checking cache in '/home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-f
                                                                                
!
  0%|          |Pushing to local               0.00/1.81k [00:00<?,     ?file/s]
 98%|█████████▊|Pushing to local          1.75k/1.80k [00:00<00:00, 17.5kfile/s]
Pushing                                                                         
1807 files pushed

--- the remote now holds the photographs ---
50M	/home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-free-traning/dvc/dvc_remote


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · رفع الصور**

بنرفع الصور للمخزن البعيد بـ `dvc push`، وبنطبع حجم المخزن البعيد عشان نشوف إنها وصلت.

</div>

Delete the whole folder, then pull it back. This is the moment DVC is actually for.

<!-- ar -->
<div dir="rtl" lang="ar">

احذف المجلد كله، ثم استرجعه. هذه اللحظة التي وُجد DVC من أجلها.

</div>

In [52]:
if DATA_ZIP:
    shutil.rmtree("photos")
    print("deleted photos/ entirely. jpg files on disk:",
          len(list(pathlib.Path(".").rglob("photos/**/*.jpg"))))

    !dvc pull -q
    print("after dvc pull      :", len(list(pathlib.Path("photos").rglob("*.jpg"))),
          "jpg files restored")

deleted photos/ entirely. jpg files on disk: 0
after dvc pull      : 1800 jpg files restored


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · حذف الصور واسترجاعها**

بنحذف مجلد `photos` بالكامل وبنعدّ الصور: صفر. بعدين `dvc pull` وبنعدّ من جديد، والصور كلها رجعت.

</div>

### What just happened

* `photos/` is **not** in Git — check `.gitignore`. Git stores `photos.dvc`, five lines of
  text, and the 43 MB of images live in DVC storage.
* A teammate clones the repo and runs `dvc pull` to get the photographs.
* The dataset now has a version. Add more photos, `dvc add photos` again, commit, and you can
  return to today's exact set forever with `git checkout` + `dvc checkout`.

### One honest exception

`data.zip` itself **is** committed to this training repo, so that the tutorial works the moment
you clone it. That is a deliberate exception for teaching, not a recommendation — it is exactly
the thing the last 50 slides argue against, and it costs every clone 39 MB forever.

In a real project you would `dvc add data.zip` (or skip the zip and track `photos/` directly),
push the bytes to a DVC remote, and let Git carry the pointer. That is what you just did in
Steps 5.1–5.3.

> And if a big file has already been committed by mistake, deck slide 47 shows how to remove
> it from history.

<!-- ar -->
<div dir="rtl" lang="ar">

**ماذا حدث للتو**

* `photos/` **ليس** في Git، تحقق من `.gitignore`. Git يحفظ `photos.dvc`، خمسة أسطر نصية، والصور موجودة في تخزين DVC.
* الزميل ينسخ المستودع ويشغّل `dvc pull` ليحصل على الصور.
* البيانات لها نسخة الآن. أضف صوراً، و `dvc add photos` مرة أخرى، واحفظ، وتستطيع العودة لمجموعة اليوم بالضبط للأبد بـ `git checkout` مع `dvc checkout`.

**استثناء صادق واحد**

`data.zip` نفسه **محفوظ** في مستودع التدريب هذا، حتى يعمل الدرس لحظة نسخه. هذا استثناء مقصود للتعليم، وليس توصية؛ إنه بالضبط ما يحذّر منه العرض، ويكلّف كل نسخة ٣٩ ميغابايت للأبد.

في مشروع حقيقي تشغّل `dvc add data.zip` (أو تتجاوز الملف المضغوط وتتتبّع `photos/` مباشرة)، وترفع البايتات إلى مخزن DVC بعيد، وتترك Git يحمل المؤشر. وهذا ما فعلته في الخطوات ٥٫١ إلى ٥٫٣.

> وإذا حُفظ ملف كبير في git بالخطأ، فالعرض يشرح كيف تزيله من التاريخ.

</div>

> That tag is the release. From another repo a consumer fetches it by name and version:
> `dvc artifacts get <repo-url> churn-classifier --rev v1.0.0` — deck slide 53.

<!-- ar -->
<div dir="rtl" lang="ar">

> هذا الوسم هو الإصدار. من مستودع آخر يجلبه المستخدم بالاسم والنسخة: `dvc artifacts get <repo-url> churn-classifier --rev v1.0.0`.

</div>

---
# Reference — worth knowing, not demonstrated here

These need a team, a cloud bucket or a second repo, so they are explained on the deck rather
than run here. Each line names the slide that covers it.

| Topic | One-line version |
|---|---|
| Data registries | `dvc import` / `dvc get` / `dvc update` — pin another repo's dataset by revision |
| Templated & matrix stages | `foreach:` and `matrix:` generate many stages from one block (you ran `foreach` in 4.4) |
| Experiments at scale | `dvc exp run --queue` + `dvc queue start -j 4`, then `dvc exp branch` to promote |
| Shared team cache | `dvc cache dir /mnt/shared` + `cache.shared group` — one copy per server |
| Plots for reviewers | a `plots:` block in `dvc.yaml`, then `dvc plots diff main workspace` |
| Migrating an existing mess | `git filter-repo` to purge a committed big file, then `dvc add` it properly |
| **Credentials** | `dvc remote modify --local` → `.dvc/config.local`, which is git-ignored. **Never put a key in `.dvc/config`** |
| Merge conflicts on data | `.gitattributes` + the `merge=dvc` driver; resolves append-only dirs, refuses real conflicts |
| The rest of `dvc.yaml` | `vars`, `wdir`, `frozen`, `always_changed`, `persist`, `push`, nested params |
| DVC 2.x → 3.x | cache moved to `.dvc/cache/files/md5/…`; `dvc cache migrate`; 2.x cannot read 3.x lock files |

<!-- ar -->
<div dir="rtl" lang="ar">

**مرجع: يستحق المعرفة، ولم نجرّبه هنا**

هذه تحتاج فريقاً، أو تخزيناً سحابياً، أو مستودعاً ثانياً، لذلك يشرحها العرض بدل تشغيلها هنا.

| الموضوع | بسطر واحد |
|---|---|
| سجلات البيانات | `dvc import` / `dvc get` / `dvc update`: ثبّت بيانات مستودع آخر على نسخة معينة |
| مراحل بقوالب ومصفوفات | `foreach:` و `matrix:` تولّد مراحل كثيرة من كتلة واحدة (شغّلت `foreach` في ٤٫٤) |
| تجارب على نطاق واسع | `dvc exp run --queue` مع `dvc queue start -j 4`، ثم `dvc exp branch` للترقية |
| ذاكرة مؤقتة مشتركة للفريق | `dvc cache dir /mnt/shared` مع `cache.shared group`: نسخة واحدة لكل خادم |
| رسوم للمراجعين | كتلة `plots:` في `dvc.yaml`، ثم `dvc plots diff main workspace` |
| ترتيب فوضى موجودة | `git filter-repo` لإزالة ملف كبير محفوظ، ثم `dvc add` له بشكل صحيح |
| **الصلاحيات** | `dvc remote modify --local` ← `.dvc/config.local`، وهو مستثنى من git. **لا تضع مفتاحاً في `.dvc/config` أبداً** |
| تعارضات الدمج على البيانات | `.gitattributes` مع مشغّل `merge=dvc`؛ يحل المجلدات التي تُضاف إليها فقط، ويرفض التعارضات الحقيقية |
| بقية `dvc.yaml` | `vars`، و `wdir`، و `frozen`، و `always_changed`، و `persist`، و `push`، والمعاملات المتداخلة |
| DVC 2.x ← 3.x | الذاكرة المؤقتة انتقلت إلى `.dvc/cache/files/md5/…`؛ `dvc cache migrate`؛ و 2.x لا يقرأ ملفات قفل 3.x |

</div>

---
# Recap

| You wanted to… | Command |
|---|---|
| put a big file under version control | `dvc add data/raw.csv` then `git add data/raw.csv.dvc` |
| see what changed | `dvc status` (`-c` compares against the remote) · `dvc data status` |
| share the data | `dvc remote add -d storage <url>` · `dvc push` / `dvc pull` |
| return to an old version | `git checkout <sha>` **then** `dvc checkout` |
| read an old version from Python | `dvc.api.read(path, repo=".", rev="HEAD~1")` |
| record how outputs are produced | `dvc.yaml` + `dvc repro` |
| see the stage graph | `dvc dag` |
| generate many similar stages | `foreach:` / `matrix:` in `dvc.yaml` |
| check if the model improved | `dvc metrics show` / `dvc metrics diff <rev>` |
| try ideas without committing | `dvc exp run -S k=v` · `dvc exp show` · `dvc queue start` |
| reuse a dataset across repos | `dvc import` / `dvc get` / `dvc update` |
| free disk space | `dvc gc -w` (careful) |
| log metrics from Python | `dvclive`: `Live()`, `log_metric`, `next_step` |
| skip work already done | the run cache — `dvc repro --no-run-cache` to force |
| store credentials safely | `dvc remote modify --local` (→ `.dvc/config.local`) |
| auto-resolve data merges | `.gitattributes` + `merge=dvc` driver |
| name a model for consumers | `artifacts:` in `dvc.yaml` + `dvc artifacts get` |
| untrack / rename / pin | `dvc remove` · `dvc move` · `dvc freeze` |

<!-- ar -->
<div dir="rtl" lang="ar">

**الخلاصة**

| تريد أن… | الأمر |
|---|---|
| تضع ملفاً كبيراً تحت الإصدارات | `dvc add data/raw.csv` ثم `git add data/raw.csv.dvc` |
| ترى ما تغيّر | `dvc status` (`-c` يقارن مع المخزن البعيد) · `dvc data status` |
| تشارك البيانات | `dvc remote add -d storage <url>` · `dvc push` / `dvc pull` |
| ترجع إلى نسخة قديمة | `git checkout <sha>` **ثم** `dvc checkout` |
| تقرأ نسخة قديمة من Python | `dvc.api.read(path, repo=".", rev="HEAD~1")` |
| تسجّل كيف تُنتج المخرجات | `dvc.yaml` مع `dvc repro` |
| ترى رسم المراحل | `dvc dag` |
| تولّد مراحل متشابهة كثيرة | `foreach:` / `matrix:` في `dvc.yaml` |
| تعرف هل تحسّن النموذج | `dvc metrics show` / `dvc metrics diff <rev>` |
| تجرّب أفكاراً دون حفظ | `dvc exp run -S k=v` · `dvc exp show` · `dvc queue start` |
| تعيد استخدام بيانات بين المستودعات | `dvc import` / `dvc get` / `dvc update` |
| تحرر مساحة القرص | `dvc gc -w` (بحذر) |
| تسجّل المقاييس من Python | `dvclive`: `Live()`، و `log_metric`، و `next_step` |
| تتجاوز عملاً منجزاً | ذاكرة التشغيل، و `dvc repro --no-run-cache` للإجبار |
| تحفظ الصلاحيات بأمان | `dvc remote modify --local` (← `.dvc/config.local`) |
| تحل دمج البيانات تلقائياً | `.gitattributes` مع مشغّل `merge=dvc` |
| تسمّي نموذجاً للمستخدمين | `artifacts:` في `dvc.yaml` مع `dvc artifacts get` |
| تلغي التتبّع / تعيد التسمية / تثبّت | `dvc remove` · `dvc move` · `dvc freeze` |

</div>

## The two rules that prevent every beginner bug

1. **Data changed?** `dvc add` **then** `git add <file>.dvc` — always both, in that order.
2. **Moved in history?** `git checkout` **then** `dvc checkout` — always both, in that order.

<!-- ar -->
<div dir="rtl" lang="ar">

**القاعدتان اللتان تمنعان كل أخطاء المبتدئين**

1. **تغيّرت البيانات؟** `dvc add` **ثم** `git add <file>.dvc`، دائماً الاثنان، وبهذا الترتيب.
2. **تحركت في التاريخ؟** `git checkout` **ثم** `dvc checkout`، دائماً الاثنان، وبهذا الترتيب.

</div>

## What to do at work tomorrow

1. `dvc init` + `dvc add` on **one** dataset, pushed to a shared folder or bucket. That alone
   kills the `data_final_v3/` folder.
2. Move your steps into `dvc.yaml` the first time you re-run something by hand.
3. Add `dvc metrics diff` to CI so reviewers see the score change on every pull request —
   see `07-dvc/cml-example.yaml` in this repo for a working one.
4. Only then: `dvc exp`, plots, shared caches.

## Docs

* DVC — <https://dvc.org/doc>
* Command reference — <https://dvc.org/doc/command-reference>
* CML, DVC in CI — <https://cml.dev>

## Cleanup (optional)

Uncomment and run to delete the sandbox and the fake remote. Leaving it is fine too —
`dvc_demo/` is a real git+DVC repo you can keep poking at.

<!-- ar -->
<div dir="rtl" lang="ar">

**ماذا تفعل في العمل غداً**

1. `dvc init` مع `dvc add` على بيانات **واحدة**، مرفوعة إلى مجلد مشترك أو تخزين سحابي. هذا وحده ينهي مجلد `data_final_v3/`.
2. انقل خطواتك إلى `dvc.yaml` أول مرة تعيد فيها تشغيل شيء يدوياً.
3. أضف `dvc metrics diff` إلى CI حتى يرى المراجعون تغيّر النتيجة في كل pull request، وملف `cml-example.yaml` في مجلد DVC بهذا المستودع مثال يعمل.
4. بعد ذلك فقط: `dvc exp`، والرسوم، والذاكرة المشتركة.

**التوثيق**

روابط توثيق DVC ومرجع الأوامر و CML موجودة في النص الإنجليزي أعلاه.

**التنظيف (اختياري)**

أزل علامة التعليق وشغّل لحذف مجلد التجربة والمخزن الوهمي. وتركه مقبول أيضاً، فـ `dvc_demo/` مستودع git و DVC حقيقي تستطيع التجربة عليه.

</div>

In [53]:
# import shutil, os
# os.chdir(BASE)
# shutil.rmtree(PROJ, ignore_errors=True)
# shutil.rmtree(REMOTE, ignore_errors=True)
# print("sandbox removed")

<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · التنظيف (اختياري)**

كل الأسطر معلّقة عمداً. لو شلت علامة التعليق، بترجع للمجلد الأصلي وبتمسح مجلد `dvc_demo` ومجلد `dvc_remote`. ولو تركتها، بيضل المستودع موجود تجرب عليه.

</div>